# Exercise 3. Aperiodic Boundaries

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from helper import Grid2DParams, AperiodicGridIntegrator

params3 = Grid2DParams(m=50)

velocity = np.zeros((400, 2))

net_aperiodic = AperiodicGridIntegrator(
    params=params3,
    boundary_mode="none",
)

out_31 = net_aperiodic.simulate(
    velocity=velocity,
    seed=0,
    store_history=True,
)

plt.figure(figsize=(5, 4))
plt.imshow(out_31["final_rate_total"], origin="lower", cmap="viridis")
plt.colorbar(label="total firing rate")
plt.title("Ex 3.1: Aperiodic boundary, no correction")
plt.xlabel("x")
plt.ylabel("y")
plt.tight_layout()
plt.show()

KeyboardInterrupt: 

In [ ]:
from helper import radial_envelope

env = radial_envelope(m=50, gamma=0.08, fratio=0.2)

plt.figure(figsize=(5, 4))
plt.imshow(env, origin="lower", cmap="viridis")
plt.colorbar(label="envelope e(x, y)")
plt.title("Ex 3.2: radial masking envelope")
plt.xlabel("x")
plt.ylabel("y")
plt.tight_layout()
plt.show()

In [ ]:
net_rate_env = AperiodicGridIntegrator(
    params=params3,
    boundary_mode="rate_envelope",
)

out_33 = net_rate_env.simulate(
    velocity=velocity,
    seed=0,
    store_history=True,
)

plt.figure(figsize=(5, 4))
plt.imshow(out_33["final_rate_total"], origin="lower", cmap="viridis")
plt.colorbar(label="total firing rate")
plt.title("Ex 3.3: Aperiodic + firing-rate envelope")
plt.xlabel("x")
plt.ylabel("y")
plt.tight_layout()
plt.show()

In [ ]:
net_input_env = AperiodicGridIntegrator(
    params=params3,
    boundary_mode="input_envelope",
)

out_34 = net_input_env.simulate(
    velocity=velocity,
    seed=0,
    store_history=True,
)

plt.figure(figsize=(5, 4))
plt.imshow(out_34["final_rate_total"], origin="lower", cmap="viridis")
plt.colorbar(label="total firing rate")
plt.title("Ex 3.4: Aperiodic + input envelope")
plt.xlabel("x")
plt.ylabel("y")
plt.tight_layout()
plt.show()

In [ ]:
params3 = Grid2DParams(m=50)

velocity, angles, segment_starts = stepwise_direction_velocity(
    warmup_steps=400,
    n_segments=12,
    segment_steps=150,
    speed=1.5,
    seed=2,
)

net_36 = AperiodicGridIntegrator(
    params=params3,
    boundary_mode="input_envelope",
)

out_36 = net_36.simulate(
    velocity=velocity,
    seed=0,
    store_history=True,
)

rate_total = out_36["rate_total"]

In [ ]:
t = np.arange(len(velocity))

plt.figure(figsize=(8, 3))
plt.plot(t, velocity[:, 0], label="vx")
plt.plot(t, velocity[:, 1], label="vy")
plt.axvline(400, color="k", linestyle="--", alpha=0.5, label="velocity onset")
plt.xlabel("time step / ms")
plt.ylabel("input velocity")
plt.title("Ex 3.6: step-wise changing input velocity")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
from scipy.signal import correlate2d


def crop_center(img, crop_size=36):
    """
    Crop central region to reduce boundary effects.
    """
    m = img.shape[0]
    start = (m - crop_size) // 2
    end = start + crop_size
    return img[start:end, start:end]


def estimate_shift_by_correlation(img1, img2, max_shift=5, crop_size=36):
    """
    Estimate how much img2 is shifted relative to img1.

    Returns:
        dx, dy in neuron-index units.
    """
    a = crop_center(img1, crop_size=crop_size)
    b = crop_center(img2, crop_size=crop_size)

    # remove mean to focus on pattern structure
    a = a - a.mean()
    b = b - b.mean()

    corr = correlate2d(b, a, mode="same")

    cy, cx = np.array(corr.shape) // 2

    local = corr[
        cy - max_shift : cy + max_shift + 1,
        cx - max_shift : cx + max_shift + 1,
    ]

    peak_y, peak_x = np.unravel_index(np.argmax(local), local.shape)

    dy = peak_y - max_shift
    dx = peak_x - max_shift

    return dx, dy


def estimate_velocity_from_grid_drift(
    rate_total,
    segment_starts,
    segment_steps=150,
    lag=20,
    discard_steps=30,
    max_shift=5,
    crop_size=36,
):
    """
    Estimate one vx, vy per velocity segment.
    """
    estimates = []

    for start in segment_starts:
        shifts = []

        # avoid first few timesteps after velocity switch
        t0 = int(start + discard_steps)
        t1 = int(start + segment_steps - lag)

        for t in range(t0, t1, lag):
            dx, dy = estimate_shift_by_correlation(
                rate_total[t],
                rate_total[t + lag],
                max_shift=max_shift,
                crop_size=crop_size,
            )
            shifts.append([dx / lag, dy / lag])

        estimates.append(np.mean(shifts, axis=0))

    return np.array(estimates)

In [ ]:
estimated_v = estimate_velocity_from_grid_drift(
    rate_total,
    segment_starts,
    segment_steps=150,
    lag=20,
    discard_steps=30,
    max_shift=5,
    crop_size=36,
)

true_v = velocity[segment_starts]

print("true velocity:")
print(true_v)

print("estimated velocity:")
print(estimated_v)

In [ ]:
plt.figure(figsize=(5, 5))
plt.scatter(true_v[:, 0], estimated_v[:, 0], label="vx")
plt.scatter(true_v[:, 1], estimated_v[:, 1], label="vy")

lims = [
    min(true_v.min(), estimated_v.min()),
    max(true_v.max(), estimated_v.max()),
]
plt.plot(lims, lims, "k--", alpha=0.5)

plt.xlabel("true input velocity")
plt.ylabel("estimated drift velocity")
plt.title("Ex 3.6: true vs estimated velocity")
plt.legend()
plt.axis("equal")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 6))

origin_x = np.arange(len(true_v))
zeros = np.zeros(len(true_v))

plt.quiver(
    origin_x,
    zeros,
    true_v[:, 0],
    true_v[:, 1],
    angles="xy",
    scale_units="xy",
    scale=1,
    label="true",
)

plt.quiver(
    origin_x,
    zeros - 0.5,
    estimated_v[:, 0],
    estimated_v[:, 1],
    angles="xy",
    scale_units="xy",
    scale=1,
    alpha=0.7,
    label="estimated",
)

plt.axhline(0, color="gray", linewidth=0.5)
plt.axhline(-0.5, color="gray", linewidth=0.5)
plt.xlabel("velocity segment")
plt.ylabel("vector display offset")
plt.title("True vs estimated velocity vectors")
plt.tight_layout()
plt.show()

In [ ]:
time_points = [400, 550, 700, 850, 1000, 1300, len(rate_total) - 1]

fig, axes = plt.subplots(1, len(time_points), figsize=(3 * len(time_points), 3))

for ax, tp in zip(axes, time_points):
    im = ax.imshow(rate_total[tp], origin="lower", cmap="viridis")
    ax.set_title(f"t = {tp}")
    ax.axis("off")

fig.colorbar(im, ax=axes, shrink=0.7)
plt.suptitle("Ex 3.6: grid pattern under step-wise changing velocity")
plt.tight_layout()
plt.show()